# Dia 6 — Agulha no vazamento 🔍

**Linux para OSINT — 21 dias** · baseado no curso de [cipher387](https://github.com/cipher387/linux-for-OSINT-21-day) (MIT)

📘 Texto completo do dia: [dia-06-grep](https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias/tree/main/dias/dia-06-grep)

## 🎬 Missão

> Chegaram dois arquivos de uma vez: a caixa de denúncias do Núcleo, com mil e
> quinhentas mensagens de vítimas, e — presente de uma fonte anônima — o log do
> "suporte" da PixVelox, com outras mil e quinhentas linhas.
>
> — Não leia — diz Bento, sem tirar o fone. — Filtra.
>
> Num post-it, a Mestra Iara deixou só uma frase: *"Golpista gosta de
> subdomínio."*

**Sua tarefa:** extrair os e-mails do log do falso suporte e encontrar o único
endereço que vive num subdomínio escondido de `pixvelox.example`. A parte
antes do `@` é a flag: `VAZANTE{...}`.

## ⚙️ Preparação

Rode a célula abaixo **uma vez**. Ela baixa o repositório para dentro do Colab e define a variável `L4O_HOME`, que todas as outras células usam.

> No Colab você é `root` e o `apt` funciona. Cada célula `%%bash` roda num processo separado: variáveis e `cd` não passam de uma célula para outra.

In [ ]:
import os, subprocess
from pathlib import Path

# Se já estamos dentro do repositório (Codespaces, máquina local), usa ele.
aqui = Path.cwd()
candidatos = [aqui, *aqui.parents]
raiz = next((p for p in candidatos if (p / 'lib' / 'comum.sh').exists()), None)

if raiz is None:  # Colab: clona o repositório
    raiz = Path('/content/linux-para-osint-21-dias')
    if not raiz.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ridd1kulusC0d3r/linux-para-osint-21-dias', str(raiz)], check=True)

os.environ['L4O_HOME'] = str(raiz)
os.environ['L4O_CORES'] = '1'
print('Repositório em', raiz)

### O tamanho do palheiro

```text
Antes de procurar, meça. `wc -l` conta linhas. Ninguém lê 3 mil linhas:
a gente filtra.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Quantas linhas temos?"
wc -l desafio/denuncias.txt desafio/log-suporte.txt
echo "---"
head -n 3 desafio/denuncias.txt

### Busca simples: grep -F

```text
`grep` imprime só as linhas que contêm o texto procurado.
  -F  trata o texto como texto fixo (sem regex) — mais rápido e sem surpresas
Aqui: quantas denúncias citam o domínio do golpe?
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Linhas com 'pixvelox' (sensível a maiúsculas)"
grep -F pixvelox desafio/denuncias.txt | head -n 5
echo "Total: $(grep -cF pixvelox desafio/denuncias.txt)"     # -c conta em vez de mostrar

### Ignorando maiúsculas: -i

```text
Golpista escreve "PixVelox", "PIXVELOX", "pixvelox"... `-i` pega todos.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Agora ignorando maiúsculas"
echo "Sem -i: $(grep -cF pixvelox desafio/denuncias.txt)"
echo "Com -i: $(grep -ciF pixvelox desafio/denuncias.txt)"

### Onde está? Número da linha: -n

```text
`-n` mostra em qual linha está cada resultado. Ótimo para citar no relatório.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Denúncias com remetente do suporte, com número da linha"
grep -n "remetente: suporte@" desafio/denuncias.txt | head -n 5

### Classes de caracteres: '[0-9]'

```text
Entre colchetes você diz "qualquer um destes". [0-9] = qualquer dígito.
Aqui: anotações que contêm algum número.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Anotações com dígitos"
grep '[0-9]' desafio/anotacoes/*.txt || dica "nenhuma linha com dígito nas anotações"
grep -c '[0-9]' desafio/log-suporte.txt

### Vários arquivos e pastas: *.txt e -r

```text
Com vários arquivos, o grep mostra o nome de cada um antes da linha.
  *.txt      todos os .txt da pasta (o Bash expande)
  -r         desce por todas as subpastas
  -l         só o NOME dos arquivos que têm o termo
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Quem fala de LGPD nas anotações?"
grep -n "LGPD" desafio/anotacoes/*.txt
echo "---"
grep -rl "subdomínio" desafio/

### Achando arquivos pelo nome: ls -R | grep

```text
grep também filtra a saída de OUTROS comandos (o "pipe" |).
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Arquivos .txt em qualquer subpasta"
# shellcheck disable=SC2010  # ls | grep de propósito: é o exemplo do curso original
ls -R desafio | grep ".txt"

### Invertendo: -v

```text
`-v` mostra o que NÃO combina. Útil para tirar ruído.
Aqui: ações do log que não são "abriu atendimento" nem "encerrou".
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Ações relevantes do falso suporte"
grep -v "abriu atendimento" desafio/log-suporte.txt | grep -v "encerrou" | head -n 5

### Regex estendida: extraindo e-mails com -E -o

```text
`-E` liga as expressões regulares estendidas (+, {2,6}, | ...).
`-o` imprime só o pedaço que combinou, não a linha inteira.
Lendo o padrão de e-mail por partes:
  [A-Za-z0-9._%+-]+   um ou mais caracteres válidos antes do @
  @                   o arroba
  [A-Za-z0-9.-]+      domínio (pode ter pontos e hífens)
  \.[A-Za-z]{2,}     ponto + terminação de 2 ou mais letras

Pegadinha: muita regex por aí usa {2,6}. Terminações novas têm mais de
6 letras (.example tem 7, .technology tem 10) e seriam CORTADAS.
Veja a diferença no bloco seguinte.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "E-mails extraídos do log (amostra)"
grep -Eo "[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}" desafio/log-suporte.txt | head -n 5

### A pegadinha do {2,6}

```text
O mesmo e-mail com as duas regex. Com {2,6} o final some.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "{2,6} corta terminações longas"
linha="contato: agente@cofre.exemplo.example"
echo "$linha" | grep -Eo "[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,6}"
echo "$linha" | grep -Eo "[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

### Contando e-mails únicos

```text
Encadeando: extrair -> ordenar -> remover repetidos -> contar.
(sort e uniq aparecem com calma no Dia 10.)
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Quantos e-mails diferentes existem em cada arquivo?"
for arq in desafio/denuncias.txt desafio/log-suporte.txt; do
  qtd="$(grep -Eo "[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}" "$arq" | sort -u | wc -l)"
  echo "$arq: $qtd e-mails únicos"
done

### Só os domínios

```text
Uma regex diferente: tudo depois do @. Depois contamos quantos endereços
há em cada domínio. Repare nos domínios parecidos com o do golpe.
```

In [ ]:
%%bash
cd "$L4O_HOME/dias/dia-06-grep" && source "$L4O_HOME/lib/comum.sh"
titulo "Domínios no log do suporte"
grep -Eo "@[A-Za-z0-9.-]+\.[A-Za-z]{2,}" desafio/log-suporte.txt | sort | uniq -c | sort -rn

## 🐍 Ponte Python

O mesmo resultado só com a biblioteca padrão do Python:

In [ ]:
!cd "$L4O_HOME/dias/dia-06-grep" && python3 equivalente.py

## 🧩 Desafio

Encontrou a flag? Troque o texto abaixo e rode a célula.

Dicas e passo a passo estão no README do dia.

In [ ]:
flag = 'VAZANTE{troque_aqui}'  # ✏️ cole sua resposta
!cd "$L4O_HOME/dias/dia-06-grep" && bash checar.sh "{flag}"

---

📝 Anote o que aprendeu no `DIARIO.md` do repositório.

Feito por [Deivison Lourenço](https://github.com/Ridd1kulusC0d3r) · curso original de [cipher387 / Cyber Detective](https://cybdetective.com)